# LLM-as-a-Judge: 1-3 Scale, 3 Judge Models

Evaluates all analogies with a **1-3 scale** judge across **3 diverse LLMs** to reduce single-model bias.

**Judge models:** GPT-4.1-mini · Gemini-2.5-flash-lite · DeepSeek-R1  
**Modes per model:** `3scale` (no few-shot) · `3scale_fewshot` (10 calibration examples)  
**Temperature:** `0.1` uniformly for all three models

**To run the full evaluation — use the PowerShell scripts:**
```
# Run all 3 models in parallel (3 separate terminal windows):
.\scripts\run_all_judges.ps1

# Run a single model (both modes, sequential):
.\scripts\run_single_judge.ps1 -Model "gpt-4.1-mini"

# Quick test (5 records, no files written):
.\scripts\run_single_judge.ps1 -Model "gpt-4.1-mini" -Test
```

**Output files** (`results/upgraded_llm/`):
- `upgraded_judge_3scale_{model}.csv` — no few-shot scores
- `upgraded_judge_3scale_fewshot_{model}.csv` — few-shot calibrated scores

**This notebook** shows the judge prompts for reference, runs a quick in-notebook test, and summarizes completed results.

In [ ]:
# =============================================================================
# CELL 1 — Imports & Paths (for notebook use only)
# =============================================================================
import sys
import os
import pandas as pd

# Output directory (relative to this notebook's location)
OUTPUT_DIR = os.path.join('..', 'results', 'upgraded_llm')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

JUDGE_MODELS = ["gpt-4.1-mini", "gemini-2.5-flash-lite", "deepseek-r1"]
JUDGE_MODES  = ["3scale", "3scale_fewshot"]
print(f"Judge models: {JUDGE_MODELS}")
print(f"Judge modes:  {JUDGE_MODES}")

In [ ]:
# =============================================================================
# CELL 2 — Judge Prompt Reference: 3-Scale Rubric (no few-shot)
# =============================================================================
# This is the prompt used in mode "3scale". For inspection only.
# The full implementation lives in core/run_judge.py.

JUDGE_INSTRUCTIONS_3SCALE = """You are an expert evaluator of scientific analogies.

Given a target concept and a chosen source analogy, evaluate whether this is a good analogy.
A good analogy uses a FAMILIAR source concept to explain an UNFAMILIAR target concept through
meaningful structural or functional parallels.

For each of the three dimensions below, first provide a brief reasoning explaining your
assessment, then give the numeric score (1, 2, or 3).

ANALOGY_COHERENCE: Does the pairing make intuitive sense?
- 3: The connection is immediately clear and natural — most people would see it without explanation.
     The source and target share an obvious structural or functional parallel.
- 2: A meaningful connection exists but requires some explanation to see.
     The link is real but not self-evident; a sentence or two is needed to establish it.
- 1: No meaningful connection exists, or the pairing is random, forced, or misleading.

MAPPING_SOUNDNESS: Could properties/mechanisms of the source map to the target?
- 3: Rich, consistent structural or functional correspondences exist.
     Multiple source properties map precisely onto target properties.
     The mapping holds across the main components of both domains.
- 2: Some valid mappings exist, but coverage is partial.
     Core correspondences work, but important aspects of the target are not represented,
     or some mappings are approximate or strained.
- 1: No valid mappings are possible, or the apparent mappings are entirely superficial
     or misleading. Source and target are fundamentally incompatible.

EXPLANATORY_POWER: Would this analogy help a learner understand the target?
- 3: The analogy clearly illuminates the target concept and supports correct reasoning.
     A learner could use it to predict or explain target behavior.
- 2: The analogy provides partial insight with notable limitations.
     It conveys the general idea but cannot support deeper reasoning,
     or it risks creating minor misconceptions.
- 1: The analogy fails to aid understanding and would likely confuse or mislead a learner.
"""

print(JUDGE_INSTRUCTIONS_3SCALE)

In [ ]:
# =============================================================================
# CELL 3 — Judge Prompt Reference: Few-Shot Calibration Examples
# =============================================================================
# These 10 examples are appended to JUDGE_INSTRUCTIONS_3SCALE for mode "3scale_fewshot".
# Scores are remapped from the original 1-5 to 1-3 (see plan for full mapping table).
# For inspection only — full implementation in core/run_judge.py.

FEW_SHOT_EXAMPLES_3SCALE = """
CALIBRATION EXAMPLES — use these to anchor your scoring.
Score each dimension independently. The scale is 1-3 only:
  3 = strong / clearly works
  2 = partial / works with caveats
  1 = doesn't work / poor or misleading

Example 1 (scores: 3 / 3 / 3)   electric circuit → water flowing through pipes
Example 2 (scores: 3 / 2 / 3)   cell → factory
Example 3 (scores: 2 / 2 / 2)   mathematical function → machine
Example 4 (scores: 1 / 1 / 1)   neural network → human brain
Example 5 (scores: 1 / 1 / 1)   chemical reaction → a novel
Example 6 (scores: 3 / 1 / 2)   democracy → majority vote in a classroom
Example 7 (scores: 2 / 3 / 3)   atom → solar system
Example 8 (scores: 3 / 2 / 1)   photosynthesis → a solar-powered factory
Example 9 (scores: 1 / 3 / 3)   compound interest → a snowball rolling downhill
Example 10 (scores: 3 / 1 / 2)  ecosystem → a family

(Full example text with per-dimension reasoning is in core/run_judge.py)
"""

# Score pattern summary — note mixed-score examples demonstrate dimension independence:
score_summary = {
    "Ex1 electric circuit/water pipes":      "3/3/3",
    "Ex2 cell/factory":                      "3/2/3",
    "Ex3 math function/machine":             "2/2/2",
    "Ex4 neural network/brain":              "1/1/1",
    "Ex5 chemical reaction/novel":           "1/1/1",
    "Ex6 democracy/classroom vote":          "3/1/2",
    "Ex7 atom/solar system":                 "2/3/3",
    "Ex8 photosynthesis/solar factory":      "3/2/1",
    "Ex9 compound interest/snowball":        "1/3/3",
    "Ex10 ecosystem/family":                 "3/1/2",
}
import pandas as pd
scores_df = pd.DataFrame(
    [(k, v.split("/")[0], v.split("/")[1], v.split("/")[2]) for k, v in score_summary.items()],
    columns=["Example", "Coherence", "Mapping", "Explanatory"]
)
print("Few-shot calibration examples — score summary (1-3 scale):")
print(scores_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 4 — Quick In-Notebook Test
# =============================================================================
# Runs 5 random records for a chosen model/mode via subprocess.
# No files are written. Use this to verify the judge works before a full run.
#
# For the full run, use PowerShell:
#   .\scripts\run_all_judges.ps1
# =============================================================================
import subprocess, sys

TEST_MODEL = "gpt-4.1-mini"       # change to test another model
TEST_MODE  = "3scale_fewshot"      # "3scale" or "3scale_fewshot"

print(f"Running quick test: {TEST_MODEL} / {TEST_MODE}")
print("=" * 60)

result = subprocess.run(
    [
        sys.executable,
        os.path.join("..", "core", "run_judge.py"),
        "--model", TEST_MODEL,
        "--mode",  TEST_MODE,
        "--test",
    ],
    # stdout/stderr stream directly to notebook output
)
print(f"\nProcess exited with code: {result.returncode}")

In [ ]:
# =============================================================================
# CELL 5 — Summary: Results from All 3 Judge Models
# =============================================================================
# Reads the 6 output CSVs (once scripts have completed) and prints per-model
# and per-mode mean scores.
# =============================================================================

def summarize_judge(judge_model: str, judge_mode: str):
    fpath = os.path.join(OUTPUT_DIR, f"upgraded_judge_{judge_mode}_{judge_model}.csv")
    if not os.path.exists(fpath):
        print(f"  [{judge_model}/{judge_mode}] NOT FOUND — run the scripts first")
        return None
    df = pd.read_csv(fpath)
    ok = df[df["status"] == "success"]
    n_err = (df["status"] == "error").sum()
    avg_c = ok["analogy_coherence"].mean()
    avg_m = ok["mapping_soundness"].mean()
    avg_e = ok["explanatory_power"].mean()
    avg_a = ok["average_score"].mean()
    print(f"  {judge_model:35s} | {judge_mode:20s} | n={len(ok):>6,} err={n_err:>3} "
          f"| coh={avg_c:.3f} map={avg_m:.3f} exp={avg_e:.3f} avg={avg_a:.3f}")
    return df


print(f"{'='*100}")
print(f"  LLM Judge Results Summary (1-3 scale)")
print(f"{'='*100}")
print(f"  {'Model':35s} | {'Mode':20s} | {'Records':>17s} "
      f"| {'Coherence':>9s} {'Mapping':>7s} {'Explanatory':>11s} {'Average':>7s}")
print(f"  {'-'*97}")

for mode in ["3scale", "3scale_fewshot"]:
    for model in JUDGE_MODELS:
        summarize_judge(model, mode)
    print(f"  {'-'*97}")

print(f"{'='*100}")